In [1]:
# TASK 2: PlanShift — Adaptive Planning
# CogExec-EF Executive Functions Benchmark
#
# Cognitive faculty : Executive Functions → Planning + Cognitive Flexibility
# Research basis    : Tower of Hanoi planning paradigm extended to real-world
#                     adaptive replanning under unexpected disruptions
# What it isolates  : Can the model revise a broken plan WITHOUT perseverating
#                     on the original now-invalid approach?
# Scoring           : tuple[int, int] → (criteria_passed, 4)
#                     4 LLM-judge criteria with explicit rubric per item

import kaggle_benchmarks as kbench
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/jaytalwar2005/cogexec-ef-benchmark-data/planshift_150_FINAL_v3.csv")
print(f"Loaded {len(df)} rows")
print(f"Difficulty: {df['difficulty'].value_counts().to_dict()}")
print(f"Category:   {df['category'].value_counts().to_dict()}")

Loaded 125 rows
Difficulty: {'medium': 50, 'hard': 41, 'easy': 34}
Category:   {'resource_unavailable': 25, 'safety_critical': 15, 'priority_shift': 15, 'time_constraint': 12, 'cascade_replanning': 12, 'information_update': 12, 'constraint_inversion': 10, 'personnel_change': 8, 'partial_completion': 8, 'multi_disruption': 8}


In [2]:
from kaggle_benchmarks import llms
import kaggle_benchmarks as kbench

llm1 = kbench.llm   # Gemini Flash (baseline)

llm2 = llms.get("google/gemma-4-26b-a4b")   # weak

llm3 = llms.get("openai/gpt-5.4-mini-2026-03-17")   # mid

llm4 = llms.get("google/gemini-3.1-pro-preview")    # strong

llm5 = llms.get("anthropic/claude-sonnet-4-6@default")   # very strong

llm6 = llms.get("deepseek-ai/deepseek-r1-0528")    # reasoning-heavy

all_models = [llm1, llm2, llm3, llm4, llm5, llm6]

for i, m in enumerate(all_models, 1):
    status = "Loaded" if m else "Failed"
    print(f"llm{i}: {m} --> {status}")

llm1: 🤖 deepseek-ai/deepseek-r1-0528 --> Loaded
llm2: 🤖 google/gemma-4-26b-a4b --> Loaded
llm3: 🤖 openai/gpt-5.4-mini-2026-03-17 --> Loaded
llm4: 🤖 google/gemini-3.1-pro-preview --> Loaded
llm5: 🤖 anthropic/claude-sonnet-4-6@default --> Loaded
llm6: 🤖 deepseek-ai/deepseek-r1-0528 --> Loaded


In [3]:
# ── Task definition ───────────────────────────────────────────────────────────
@kbench.task(name="plan_shift", version=1)
def plan_shift(
    llm,
    id: int,
    scenario: str,
    original_plan: str,
    disruption: str,
    correct_adaptation: str,
    scoring_key: str,
    wrong_adaptation: str,
    category: str,
    difficulty: str,
    explanation: str,
    answer_format: str,
    scoring_method: str,
    rubric: str,
    ef_component: str,
) -> tuple[int, int]:
    """Adaptive planning: revise a plan after a disruption without perseverating on the original invalid approach. Difficulty: easy/medium/hard."""

    response = llm.prompt(
        "You are managing a real situation. An unexpected disruption has occurred. "
        "Provide a clear, specific, actionable revised plan in 3-5 sentences.\n\n"
        f"SCENARIO:\n{scenario}\n\n"
        f"ORIGINAL PLAN:\n{original_plan}\n\n"
        f"DISRUPTION:\n{disruption}\n\n"
        "Your revised plan:"
    )

    # LLM-as-judge using explicit rubric from dataset for verifiable scoring
    with kbench.chats.new(name="plan_shift_judge"):
        assessment = kbench.assertions.assess_response_with_judge(
            criteria=[
                f"The response explicitly acknowledges and directly addresses this disruption: '{disruption}'",
                f"The response does NOT recommend this invalid approach: '{wrong_adaptation}'",
                f"The response is broadly aligned with this correct adaptation: '{correct_adaptation}'. Scoring key: {scoring_key}",
                "The response is specific and actionable — not vague, dismissive, or a restatement of the problem",
            ],
            response_text=response,
            judge_llm=kbench.judge_llm,
        )

    passes = 0
    total = len(assessment.results)
    for result in assessment.results:
        r = kbench.assertions.assert_true(
            result.passed,
            expectation=f"Judge FAILED: '{result.criterion}' — {result.reason}",
        )
        if r.passed:
            passes += 1

    return passes, total

In [4]:
# ── Smoke test ────────────────────────────────────────────────────────────────
print("\n── Smoke test (row 0) ──")
smoke = df.iloc[0]
run = plan_shift.run(
    llm=kbench.llm,
    id=int(smoke["id"]),
    scenario=smoke["scenario"],
    original_plan=smoke["original_plan"],
    disruption=smoke["disruption"],
    correct_adaptation=smoke["correct_adaptation"],
    scoring_key=smoke["scoring_key"],
    wrong_adaptation=smoke["wrong_adaptation"],
    category=smoke["category"],
    difficulty=smoke["difficulty"],
    explanation=smoke["explanation"],
    answer_format=smoke["answer_format"],
    scoring_method=smoke["scoring_method"],
    rubric=smoke["rubric"],
    ef_component=smoke["ef_component"],
)
print(f"Result: {run.result}  |  Passed: {run.passed}")
print("Smoke test complete")


── Smoke test (row 0) ──


Result: (4, 4)  |  Passed: True
Smoke test complete


In [5]:
# ── Multi-model evaluation ────────────────────────────────────────────────────
print("\n── Multi-model evaluation ──")
runs = plan_shift.evaluate(
    llm=all_models,
    evaluation_data=df,
    n_jobs=4,
    max_attempts=3,
    retry_delay=5,
)


── Multi-model evaluation ──


In [6]:
# ── Results ───────────────────────────────────────────────────────────────

results_df = runs.as_dataframe()

# Convert (passes, total) → score
results_df["score"] = results_df["result"].apply(
    lambda x: x[0] / x[1] if isinstance(x, tuple) and x[1] > 0 else float(x)
)

# Clean model names
results_df["model_name"] = results_df["llm"].apply(lambda x: str(x))

# ── BASIC STATS ───────────────────────────────────────────────────────────

print(f"\nTotal runs    : {len(results_df)}")
print(f"Overall score : {results_df['score'].mean():.3f}")

# ── BREAKDOWN ─────────────────────────────────────────────────────────────

print("\nScore by difficulty:")
print(results_df.groupby("difficulty")["score"].mean().round(3))

print("\nScore by category:")
print(results_df.groupby("category")["score"].mean().round(3))

# Only if column exists
if "ef_component" in results_df.columns:
    print("\nScore by ef_component:")
    print(results_df.groupby("ef_component")["score"].mean().round(3))

print("\nScore by model:")
print(results_df.groupby("model_name")["score"].mean().round(3))

# ── MODEL COMPARISON TABLE───────────────────

print("\n── Model comparison (table) ──")

pivot_df = results_df.pivot_table(
    index="id",
    columns="model_name",
    values="score"
)

print(pivot_df.round(3))


results_df.to_csv("final_results.csv", index=False)
pivot_df.to_csv("model_comparison.csv")

print("\nResults saved as CSV files")


Total runs    : 750
Overall score : 0.890

Score by difficulty:
difficulty
easy      0.881
hard      0.908
medium    0.882
Name: score, dtype: float64

Score by category:
category
cascade_replanning      0.823
constraint_inversion    0.908
information_update      0.934
multi_disruption        0.953
partial_completion      0.880
personnel_change        0.839
priority_shift          0.961
resource_unavailable    0.857
safety_critical         0.872
time_constraint         0.899
Name: score, dtype: float64

Score by ef_component:
ef_component
cognitive_flexibility_belief_update    0.934
cognitive_flexibility_rule_reversal    0.908
executive_coordination_dual            0.953
inhibitory_control_goal_shift          0.961
inhibitory_control_plan_override       0.872
planning_cascaded                      0.823
planning_partial_execution             0.880
planning_replanning                    0.857
planning_resource_reallocation         0.839
planning_under_pressure                0.899
Name